# ModernFloraBERT regression fine-tuning (Kaggle)

Run this notebook on Kaggle with internet enabled and `modernflorabert-base` attached. Also attach the output of the maize MLM run as a Kaggle input dataset, or set `FLORABERT_MAIZE_MLM_INPUT_DIR` to its mounted input directory.

The notebook copies the maize-adapted ModernBERT checkpoint, unchanged tokenizer, and NAM train/eval/test files into `/kaggle/temp/florabert_regression_inputs` by default. `/kaggle/working` is reserved for regression outputs and checkpoints.

The regression model is initialized from the copied maize-MLM checkpoint, uses the repository's mean-pooling head and natural-log target `ln(TPM + 0.001)`, evaluates every epoch, and reports the retained best validation checkpoint. By default this notebook runs the StableAdamW optimizer; set `FLORABERT_REGRESSION_OPTIMIZER=lamb` for the existing LAMB control.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


def env_bool(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.lower() in {'1', 'true', 'yes', 'y', 'on'}


kaggle_input_root = Path(
    os.environ.get('KAGGLE_INPUT_ROOT', '/kaggle/input')
).expanduser()
plant_input_root = Path(
    os.environ.get(
        'FLORABERT_MODERNFLORABERT_INPUT',
        '/kaggle/input/modernflorabert-base/florabert',
    )
).expanduser()
configured_maize_mlm_input = os.environ.get(
    'FLORABERT_MAIZE_MLM_INPUT_DIR',
    '',
).strip()
maize_mlm_input_root = (
    Path(configured_maize_mlm_input).expanduser()
    if configured_maize_mlm_input
    else None
)
genex_input_root = Path(
    os.environ.get(
        'FLORABERT_GENEX_INPUT_DIR',
        str(plant_input_root / 'data' / 'final' / 'transformer' / 'genex' / 'nam'),
    )
).expanduser()

# Inputs are copied here; this path must not be under /kaggle/working.
input_workspace = Path(
    os.environ.get(
        'FLORABERT_INPUT_WORKSPACE',
        '/kaggle/temp/florabert_regression_inputs',
    )
).expanduser()
output_root = Path(
    os.environ.get(
        'FLORABERT_OUTPUT_ROOT',
        '/kaggle/working/florabert_regression_outputs',
    )
).expanduser()

working_root = Path('/kaggle/working').resolve()
if input_workspace.resolve() == working_root or working_root in input_workspace.resolve().parents:
    raise ValueError(
        f'Input workspace must not be under /kaggle/working: {input_workspace}'
    )

repo_dir = Path(
    os.environ.get('FLORABERT_REPO_DIR', '/kaggle/temp/florabert_repo')
).expanduser()
repo_url = os.environ.get(
    'FLORABERT_REPO_URL',
    'https://github.com/gurveervirk/florabert.git',
)
repo_ref = os.environ.get(
    'FLORABERT_REPO_REF',
    'feat/modernbert-maize-mlm-ablation',
)
expected_commit = os.environ.get('FLORABERT_REPO_COMMIT', '').strip()
requested_regression_optimizer = os.environ.get(
    'FLORABERT_REGRESSION_OPTIMIZER',
    'stableadamw',
).strip().lower()
optimizer_aliases = {
    'stable_adamw': 'stableadamw',
    'stable-adamw': 'stableadamw',
}
regression_optimizer = optimizer_aliases.get(
    requested_regression_optimizer,
    requested_regression_optimizer,
)
if regression_optimizer not in {'lamb', 'adam', 'adamw', 'stableadamw'}:
    raise ValueError(
        'FLORABERT_REGRESSION_OPTIMIZER must be one of: '
        'lamb, adam, adamw, stableadamw'
    )

if not (repo_dir / '.git').is_dir():
    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise RuntimeError(
            f'{repo_dir} exists but is not an empty git checkout'
        )

    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', '--branch', repo_ref, '--depth', '1', repo_url, str(repo_dir)],
        check=True,
    )

actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'],
    cwd=repo_dir,
    text=True,
).strip()
if expected_commit and actual_commit != expected_commit:
    raise RuntimeError(
        f'Unexpected FloraBERT commit {actual_commit}; expected {expected_commit}'
    )

genex_dir = input_workspace / 'data' / 'genex' / 'nam'
tokenizer_dir = input_workspace / 'models' / 'modernbert-tokenizer'
maize_lm_checkpoint = input_workspace / 'models' / 'transformer' / 'language-model-modernbert-maize'
regression_output = output_root / f'prediction-model-modernbert-maize-{regression_optimizer}'

for path in [genex_dir, tokenizer_dir, maize_lm_checkpoint, regression_output]:
    path.mkdir(parents=True, exist_ok=True)

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))

print('Kaggle input root:', kaggle_input_root)
print('Plant input root:', plant_input_root)
print('Maize MLM input root:', maize_mlm_input_root or 'auto-discover under Kaggle inputs')
print('NAM data input root:', genex_input_root)
print('Input workspace:', input_workspace)
print('Regression output root:', output_root)
print('Repo:', repo_dir)
print('Repo ref:', repo_ref)
print('Repo commit:', actual_commit)

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-r',
        str(repo_dir / 'requirements.txt'),
    ],
    cwd=repo_dir,
)

subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'wandb>=0.19',
    ]
)

import importlib.metadata as importlib_metadata
import torch

for package_name in [
    'torch',
    'transformers',
    'datasets',
    'accelerate',
    'wandb',
    'torch-optimi',
]:
    try:
        print(package_name, importlib_metadata.version(package_name))
    except importlib_metadata.PackageNotFoundError:
        print(package_name, 'not found')

cuda_count = torch.cuda.device_count()
print('CUDA device count:', cuda_count)
if cuda_count:
    for device_idx in range(cuda_count):
        print(f'CUDA {device_idx}: {torch.cuda.get_device_name(device_idx)}')
else:
    raise RuntimeError('A Kaggle GPU runtime is required for practical regression fine-tuning.')

In [ ]:
# All experiment inputs are attached Kaggle datasets. No HF or Kaggle
# download authentication is needed in this notebook.
def runtime_secret(name):
    value = os.environ.get(name)
    return value.strip() if value and value.strip() else None


print('Using attached Kaggle inputs; no dataset download is required.')
print('W&B credentials will be loaded from the Kaggle Secret named key.')

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient


wandb_key = runtime_secret('WANDB_API_KEY')
if not wandb_key:
    try:
        user_secrets = UserSecretsClient()
        wandb_key = user_secrets.get_secret('key')
    except Exception:
        wandb_key = None

if not wandb_key:
    from getpass import getpass

    wandb_key = getpass(
        'Weights & Biases API key (hidden input; not saved in this notebook): '
    ).strip()

if not wandb_key:
    raise RuntimeError('No W&B API key was supplied.')

os.environ['WANDB_API_KEY'] = wandb_key
os.environ.setdefault(
    'WANDB_PROJECT',
    'florabert-modernbert-maize-ablation',
)
wandb.login(key=wandb_key)
del wandb_key

print('W&B authentication is ready.')
print('W&B project:', os.environ['WANDB_PROJECT'])

In [ ]:
def has_model_weights(directory):
    return any(
        path.is_file()
        for pattern in (
            '*.safetensors',
            '*.bin',
            '*.safetensors.index.json',
            '*.bin.index.json',
        )
        for path in directory.glob(pattern)
    )


def copy_input_file(source_path, destination_path):
    source_path = Path(source_path)
    destination_path = Path(destination_path)

    if not source_path.is_file() or source_path.stat().st_size == 0:
        raise FileNotFoundError(
            f'Missing or empty attached input file: {source_path}'
        )

    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if (
        not destination_path.is_file()
        or destination_path.stat().st_size != source_path.stat().st_size
    ):
        shutil.copy2(source_path, destination_path)
        print('Copied', source_path, '->', destination_path)
    else:
        print('Reusing copied file:', destination_path)


def resolve_maize_model_source():
    candidate_dirs = []

    if maize_mlm_input_root is not None:
        candidate_dirs.extend(
            [
                maize_mlm_input_root,
                maize_mlm_input_root / 'models' / 'transformer' / 'language-model-modernbert-maize',
                maize_mlm_input_root / 'florabert' / 'models' / 'transformer' / 'language-model-modernbert-maize',
            ]
        )

    if kaggle_input_root.is_dir():
        candidate_dirs.extend(
            [
                kaggle_input_root / 'modernflorabert-maize-mlm' / 'models' / 'transformer' / 'language-model-modernbert-maize',
                kaggle_input_root / 'modernflorabert-maize-mlm' / 'florabert' / 'models' / 'transformer' / 'language-model-modernbert-maize',
            ]
        )

    for candidate in candidate_dirs:
        if (candidate / 'config.json').is_file() and has_model_weights(candidate):
            return candidate

    search_roots = [maize_mlm_input_root] if maize_mlm_input_root else []
    if kaggle_input_root.is_dir():
        search_roots.append(kaggle_input_root)

    discovered = []
    for search_root in search_roots:
        if search_root is None or not search_root.is_dir():
            continue
        discovered.extend(
            config_path.parent
            for config_path in search_root.rglob('config.json')
            if 'language-model-modernbert-maize' in str(config_path.parent)
            and has_model_weights(config_path.parent)
        )

    if discovered:
        return sorted(set(discovered), key=str)[0]

    raise FileNotFoundError(
        'Could not find the maize-adapted ModernBERT checkpoint. Attach the '
        'prior MLM output as a Kaggle input or set '
        'FLORABERT_MAIZE_MLM_INPUT_DIR to its mounted directory.'
    )


def resolve_tokenizer_source():
    direct = plant_input_root / 'models' / 'modernbert-byte-level-bpe-tokenizer'
    if (direct / 'tokenizer.json').is_file():
        return direct

    if plant_input_root.is_dir():
        discovered = sorted(
            tokenizer_path.parent
            for tokenizer_path in plant_input_root.rglob('tokenizer.json')
            if 'modernbert' in str(tokenizer_path).lower()
        )
        if discovered:
            return discovered[0]

    raise FileNotFoundError(
        f'Could not find the ModernBERT tokenizer under {plant_input_root}'
    )


maize_model_source = resolve_maize_model_source()
tokenizer_source = resolve_tokenizer_source()

print('Maize MLM checkpoint source:', maize_model_source)
print('Tokenizer source:', tokenizer_source)
print('Regression data source:', genex_input_root)

if not genex_input_root.is_dir():
    raise FileNotFoundError(f'NAM data directory does not exist: {genex_input_root}')

shutil.copytree(
    maize_model_source,
    maize_lm_checkpoint,
    dirs_exist_ok=True,
)
shutil.copytree(
    tokenizer_source,
    tokenizer_dir,
    dirs_exist_ok=True,
)

for filename in ['train.tsv', 'eval.tsv', 'test.tsv']:
    copy_input_file(genex_input_root / filename, genex_dir / filename)

if not (maize_lm_checkpoint / 'config.json').is_file() or not has_model_weights(maize_lm_checkpoint):
    raise RuntimeError(
        f'Copied maize MLM checkpoint is incomplete: {maize_lm_checkpoint}'
    )
if not (tokenizer_dir / 'tokenizer.json').is_file():
    raise RuntimeError(f'Copied tokenizer is incomplete: {tokenizer_dir}')

for filename in ['train.tsv', 'eval.tsv', 'test.tsv']:
    path = genex_dir / filename
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f'Copied regression file is incomplete: {path}')

print('Copied maize MLM checkpoint:', maize_lm_checkpoint)
print('Copied tokenizer:', tokenizer_dir)
print('Copied regression data:', genex_dir)
print('Working directory reserved for output:', output_root)

In [ ]:
from transformers import AutoConfig, PreTrainedTokenizerFast

from module.florabert import config as flora_config
from module.florabert import transformers as flora_transformers
from module.florabert import utils as flora_utils


source_config = AutoConfig.from_pretrained(
    str(maize_lm_checkpoint),
    local_files_only=True,
)
regression_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    str(tokenizer_dir),
    local_files_only=True,
)

print('Exact maize MLM checkpoint consumed by regression:', maize_lm_checkpoint)
print('Checkpoint model type:', source_config.model_type)
print('Checkpoint vocab size:', source_config.vocab_size)
print('Tokenizer vocab size:', len(regression_tokenizer))

assert source_config.model_type == 'modernbert'
assert source_config.vocab_size == len(regression_tokenizer), (
    'Refusing to resize the tokenizer: checkpoint vocab '
    f'{source_config.vocab_size} != tokenizer vocab {len(regression_tokenizer)}'
)

flora_config.reload_settings()
regression_settings = flora_utils.get_model_settings(
    flora_config.settings,
    model_name='modernbert-pred-mean-pool',
)
regression_settings['output_mode'] = 'regression'
regression_settings['num_labels'] = len(flora_config.tissues)

_, loaded_tokenizer, regression_model = flora_transformers.load_model(
    'modernbert-pred-mean-pool',
    str(tokenizer_dir),
    pretrained_model=str(maize_lm_checkpoint),
    log_offset=0.001,
    **regression_settings,
)

total_params = flora_utils.count_model_parameters(
    regression_model,
    trainable_only=False,
)
print('Verified maize MLM base weights loaded into regression.')
print('Regression model parameter count:', total_params)

smoke_inputs = loaded_tokenizer(
    'ACGTACGTACGTTTTAAACCCGGG',
    return_tensors='pt',
    max_length=loaded_tokenizer.model_max_length,
    truncation=True,
    padding='max_length',
)
# The custom ModernBERT forward does not accept token_type_ids.
smoke_inputs = {
    key: value
    for key, value in smoke_inputs.items()
    if key in {'input_ids', 'attention_mask', 'position_ids'}
}
print('Smoke-test model inputs:', sorted(smoke_inputs))
regression_model.eval()
with torch.no_grad():
    smoke_outputs = regression_model(**smoke_inputs)

assert smoke_outputs.logits.shape == (1, len(flora_config.tissues))
assert torch.isfinite(smoke_outputs.logits).all()
print('One regression forward pass:', tuple(smoke_outputs.logits.shape))

del regression_model, loaded_tokenizer, smoke_outputs, smoke_inputs
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
finetune_settings = dict(flora_config.settings['training']['finetune'])
finetune_settings['optimizer'] = regression_optimizer
regression_learning_rate = (
    float(os.environ['FLORABERT_REGRESSION_LEARNING_RATE'])
    if os.environ.get('FLORABERT_REGRESSION_LEARNING_RATE')
    else None
)
regression_epochs = (
    int(os.environ['FLORABERT_REGRESSION_EPOCHS'])
    if os.environ.get('FLORABERT_REGRESSION_EPOCHS')
    else None
)
n_workers = int(os.environ.get('FLORABERT_DATA_WORKERS', '2'))
force_rerun = env_bool('FLORABERT_FORCE_RERUN', False)
disable_gradient_clipping = env_bool(
    'FLORABERT_DISABLE_GRADIENT_CLIPPING',
    regression_optimizer == 'stableadamw',
)
if disable_gradient_clipping:
    finetune_settings.pop('max_grad_norm', None)
regression_resume_from = (
    Path(os.environ['FLORABERT_REGRESSION_RESUME_FROM']).expanduser()
    if os.environ.get('FLORABERT_REGRESSION_RESUME_FROM')
    else None
)

print('Current repo regression settings:', finetune_settings)
print('Natural-log target: ln(TPM + 0.001)')
print('Effective optimizer:', regression_optimizer)
print('Effective learning rate:', regression_learning_rate or finetune_settings['learning_rate'])
print('Effective epochs:', regression_epochs or finetune_settings['num_train_epochs'])
print('Regression resume checkpoint:', regression_resume_from or 'none')
print('CUDA devices:', cuda_count)
print('Conventional gradient clipping:', 'disabled' if disable_gradient_clipping else 'enabled/configured')

In [ ]:
def launch_repo_script(relative_script, arguments):
    script_path = repo_dir / relative_script

    if cuda_count > 1:
        accelerate_exe = shutil.which('accelerate')
        if accelerate_exe:
            command = [
                accelerate_exe,
                'launch',
                '--num_processes',
                str(cuda_count),
                str(script_path),
                *map(str, arguments),
            ]
        else:
            command = [
                sys.executable,
                '-m',
                'accelerate.commands.launch',
                '--num_processes',
                str(cuda_count),
                str(script_path),
                *map(str, arguments),
            ]
    else:
        command = [sys.executable, '-u', str(script_path), *map(str, arguments)]

    environment = os.environ.copy()
    environment['PYTHONPATH'] = (
        str(repo_dir) + os.pathsep + environment.get('PYTHONPATH', '')
    )
    environment['PYTHONUNBUFFERED'] = '1'

    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(
        command,
        cwd=str(repo_dir),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        for line in process.stdout:
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


def regression_arguments():
    arguments = [
        '--model-name',
        'modernbert-pred-mean-pool',
        '--data-dir',
        str(genex_dir),
        '--train-data',
        'train.tsv',
        '--eval-data',
        'eval.tsv',
        '--test-data',
        'test.tsv',
        '--tokenizer-dir',
        str(tokenizer_dir),
        '--output-dir',
        str(regression_output),
        '--transformation',
        'log',
        '--log-offset',
        '0.001',
        '--precision',
        'fp16',
        '--n-workers',
        str(n_workers),
        '--optimizer',
        regression_optimizer,
    ]

    if regression_resume_from is None:
        arguments.extend(['--pretrained-model', str(maize_lm_checkpoint)])
    else:
        if not regression_resume_from.is_dir():
            raise FileNotFoundError(
                f'Regression resume checkpoint does not exist: {regression_resume_from}'
            )
        if not (regression_resume_from / 'training_state.pt').is_file():
            raise FileNotFoundError(
                'Regression resume requires training_state.pt alongside the '
                f'checkpoint: {regression_resume_from}'
            )
        arguments.extend(['--resume-from-checkpoint', str(regression_resume_from)])

    if regression_learning_rate is not None:
        arguments.extend(['--learning-rate', str(regression_learning_rate)])
    if regression_epochs is not None:
        arguments.extend(['--num-train-epochs', str(regression_epochs)])
    if disable_gradient_clipping:
        arguments.append('--no-grad-clipping')

    return arguments

In [ ]:
best_config = regression_output / 'best' / 'config.json'
if (
    best_config.is_file()
    and not force_rerun
    and regression_resume_from is None
):
    print(
        'Regression output already exists; set FLORABERT_FORCE_RERUN=1 '
        'to retrain:',
        regression_output,
    )
else:
    launch_repo_script(
        Path('scripts/1-modeling/finetune.py'),
        regression_arguments(),
    )

print('Validation selection is performed after every epoch by finetune.py.')
print('The retained best checkpoint is:', regression_output / 'best')

In [ ]:
import pandas as pd
from IPython.display import display

metrics_file = regression_output / 'metrics.jsonl'
best_file = regression_output / 'best_metrics.json'
best_checkpoint = regression_output / 'best'

if not metrics_file.is_file():
    raise FileNotFoundError(f'Validation history is missing: {metrics_file}')
if not best_file.is_file() or not (best_checkpoint / 'config.json').is_file():
    raise RuntimeError(
        f'Best validation checkpoint is missing under {regression_output}'
    )

records = [
    json.loads(line)
    for line in metrics_file.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
history = pd.DataFrame(
    [
        {'epoch': record['epoch'], **record['overall']}
        for record in records
    ]
)

display(
    history[
        [
            'epoch',
            'mse',
            'r2',
            'pearson_r2',
            'prediction_mean',
            'prediction_std',
            'target_mean',
            'target_std',
        ]
    ]
)
print('Best-checkpoint metadata:')
print(best_file.read_text(encoding='utf-8'))
print('Best checkpoint:', best_checkpoint)

In [ ]:
import numpy as np
from IPython.display import display
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader

from module.florabert import dataio as flora_dataio


def population_std(values):
    return float(np.asarray(values, dtype='float64').std(ddof=0))


def metric_row(scope, targets, predictions):
    targets = np.asarray(targets, dtype='float64').reshape(-1)
    predictions = np.asarray(predictions, dtype='float64').reshape(-1)
    correlation = (
        float('nan')
        if targets.std() == 0 or predictions.std() == 0
        else float(np.corrcoef(targets, predictions)[0, 1])
    )

    return {
        'scope': scope,
        'mse': float(np.mean((targets - predictions) ** 2)),
        'sklearn_r2': float(r2_score(targets, predictions)),
        'pearson_r2': correlation ** 2 if correlation == correlation else float('nan'),
        'prediction_mean': float(predictions.mean()),
        'prediction_std': population_std(predictions),
        'target_mean': float(targets.mean()),
        'target_std': population_std(targets),
    }


def report_metrics(targets, predictions):
    rows = [metric_row('overall', targets, predictions)]

    for tissue_idx, tissue in enumerate(flora_config.tissues):
        rows.append(
            metric_row(
                tissue,
                targets[:, tissue_idx],
                predictions[:, tissue_idx],
            )
        )

    return pd.DataFrame(rows).set_index('scope')


if not (best_checkpoint / 'config.json').is_file():
    raise FileNotFoundError(f'Best checkpoint is missing: {best_checkpoint}')

best_settings = dict(regression_settings)
best_settings['output_mode'] = 'regression'
best_settings['num_labels'] = len(flora_config.tissues)
_, eval_tokenizer, eval_model = flora_transformers.load_model(
    'modernbert-pred-mean-pool',
    str(tokenizer_dir),
    pretrained_model=str(best_checkpoint),
    log_offset=0.001,
    **best_settings,
)

test_datasets = flora_dataio.load_datasets(
    eval_tokenizer,
    str(genex_dir / 'test.tsv'),
    seq_key='sequence',
    file_type='csv',
    delimiter='\t',
    transformation='log',
    log_offset=0.001,
    shuffle=False,
    n_workers=n_workers,
)
test_dataset = test_datasets['train'].remove_columns(['sequence'])
eval_batch_size = int(os.environ.get('FLORABERT_EVAL_BATCH_SIZE', '8'))
test_loader = DataLoader(
    test_dataset,
    batch_size=eval_batch_size,
    collate_fn=flora_dataio.load_data_collator('pred'),
    shuffle=False,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
eval_model.to(device).eval()
predictions = []
targets = []

with torch.no_grad():
    for batch in test_loader:
        labels = batch['labels'].to(device)
        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key in {'input_ids', 'attention_mask', 'position_ids', 'labels'}
        }
        outputs = eval_model(**inputs)
        predictions.append(outputs.logits.detach().cpu())
        targets.append(labels.detach().cpu())

target_array = torch.cat(targets).numpy()
prediction_array = torch.cat(predictions).numpy()
results = report_metrics(target_array, prediction_array)
print('Test metrics for the maize-adapted best checkpoint in log space:')
display(results)

evaluation_dir = output_root / 'evaluation'
evaluation_dir.mkdir(parents=True, exist_ok=True)
(evaluation_dir / 'maize_adapted_best.json').write_text(
    json.dumps(results.reset_index().to_dict(orient='records'), indent=2),
    encoding='utf-8',
)
print('Saved metrics:', evaluation_dir / 'maize_adapted_best.json')